 # Stochastic Methods in Finance

## Importing packages

In [130]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

## Importing and inspecting the dataset

In [154]:
data = pd.read_csv("./HistoricalData_1746127004374.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2515 entries, 0 to 2514
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Date        2515 non-null   object
 1   Close/Last  2515 non-null   object
 2   Volume      2515 non-null   int64 
 3   Open        2515 non-null   object
 4   High        2515 non-null   object
 5   Low         2515 non-null   object
dtypes: int64(1), object(5)
memory usage: 118.0+ KB


In [202]:
data.head()

,Close/Last,Volume,Open,High,Low,Return
Date,,,,,,
2020-01-02,160.62,22634550,158.78,160.730,158.3300,NaN
2020-01-03,158.62,21121680,158.32,159.945,158.0600,-0.012452
2020-01-06,159.03,20826700,157.08,159.100,156.5100,0.002585
2020-01-07,157.58,21881740,159.32,159.670,157.3200,-0.009118
2020-01-08,160.09,27762030,158.93,160.800,157.9491,0.015928


In [201]:
data.tail()

,Close/Last,Volume,Open,High,Low,Return
Date,,,,,,
2025-04-24,387.30,22232290,375.695,388.45,375.190,0.034483
2025-04-25,391.85,18973170,387.000,392.16,384.600,0.011748
2025-04-28,391.16,16579430,391.955,392.74,386.638,-0.001761
2025-04-29,394.04,14973980,391.300,395.10,390.380,0.007363
2025-04-30,395.26,36461080,390.300,396.66,384.440,0.003096


## Preprocessing

In [158]:
# Converting the date column to datetime
data.index = pd.to_datetime(data.index)

# Setting the date as the index
data.sort_index(inplace=True)

# We start from the last 5 years of data
data = data[data.index >= '2020-01-01']

# Setting the variables as numeric
# Remove dollar signs and any other non-numeric characters, then convert to numeric
data['Close/Last'] = data['Close/Last'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Open'] = data['Open'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['High'] = data['High'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Low'] = data['Low'].replace({'\$': '', ',': ''}, regex=True).astype(float)
data['Volume'] = data['Volume'].replace({',': ''}, regex=True).astype(int)

# Check the types of the columns
print(data.dtypes)

Close/Last    float64
Volume          int64
Open          float64
High          float64
Low           float64
dtype: object


<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:14: SyntaxWarning: invalid escape sequence '\$'
<>:15: SyntaxWarning: invalid escape sequence '\$'
<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:13: SyntaxWarning: invalid escape sequence '\$'
<>:14: SyntaxWarning: invalid escape sequence '\$'
<>:15: SyntaxWarning: invalid escape sequence '\$'
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_83260/1054702915.py:12: SyntaxWarning: invalid escape sequence '\$'
  data['Close/Last'] = data['Close/Last'].replace({'\$': '', ',': ''}, regex=True).astype(float)
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_83260/1054702915.py:13: SyntaxWarning: invalid escape sequence '\$'
  data['Open'] = data['Open'].replace({'\$': '', ',': ''}, regex=True).astype(float)
/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_83260/1054702915.py:14: SyntaxWarning: invalid escape sequence '\$'
  data['High'] = data['High']

In [161]:
data.head()

,Close/Last,Volume,Open,High,Low
Date,,,,,
2020-01-02,160.62,22634550,158.78,160.730,158.3300
2020-01-03,158.62,21121680,158.32,159.945,158.0600
2020-01-06,159.03,20826700,157.08,159.100,156.5100
2020-01-07,157.58,21881740,159.32,159.670,157.3200
2020-01-08,160.09,27762030,158.93,160.800,157.9491


In [160]:
data.tail()

,Close/Last,Volume,Open,High,Low
Date,,,,,
2025-04-24,387.30,22232290,375.695,388.45,375.190
2025-04-25,391.85,18973170,387.000,392.16,384.600
2025-04-28,391.16,16579430,391.955,392.74,386.638
2025-04-29,394.04,14973980,391.300,395.10,390.380
2025-04-30,395.26,36461080,390.300,396.66,384.440


## Tasks

### 1. Creating the binomial tree

In [177]:
# Calculate daily returns and the standard deviation (volatility)
data['Return'] = data['Close/Last'].pct_change()
sigma_daily = data['Return'].std()  # Sample standard deviation of daily returns
sigma_annual = sigma_daily * np.sqrt(250)  # Annualize the volatility (assuming 250 trading days)

# Initial stock price S0 (price on April 28, 2025)
S_0 = data.loc['2025-04-28', 'Close/Last']  # Update with the correct date or your preferred method

# Given parameters
n = 25  # Number of periods
r = 0.01  # Risk-free rate (1% per annum)
r_daily = (1 + r) ** (1/250) - 1  # Daily risk-free rate
T = 25/250 # Time to maturity in years (25 days, since we are predicting the price in 25 days)

# Calculate the up and down factors
dt = T / n  # Time step
u = np.exp(sigma_annual * np.sqrt(dt))  # Up factor
d = np.exp(-sigma_annual * np.sqrt(dt))  # Down factor

# Probability
p = 0.5

/var/folders/wn/nf_gxvtd0zsfbsgzc8y_qpn40000gn/T/ipykernel_83260/397631103.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Return'] = data['Close/Last'].pct_change()


In [164]:
# Binomial tree construction
# Initialize the binomial tree as a list of lists
tree = []

# Root node is S_0
tree.append([S_0])

# Fill the tree with prices
for i in range(1, n + 1):  # Loop through each level
    level = []
    for j in range(2 ** i):  # Each level has 2^i nodes
        # Calculate the price for each node
        up_moves = bin(j).count('1')  # Count of '1's in binary representation gives up moves
        down_moves = i - up_moves
        price = S_0 * (u ** up_moves) * (d ** down_moves)
        level.append(price)
    tree.append(level)

# Number of terminal nodes
num_terminal_nodes = len(tree[-1])

# Print the number of terminal nodes
print(f"Number of terminal nodes: {num_terminal_nodes}")

Number of terminal nodes: 33554432


### 2. Average prices for each node

In [165]:
# Initialize a list to store average prices at terminal nodes
average_prices = []

# Loop through each terminal path (there are 2^n paths)
for j in range(2 ** n):
    # Determine the up/down path as a binary string
    path = bin(j)[2:].zfill(n)  # Pad to ensure length n
    prices = [S_0]  # Start with the initial price

    current_price = S_0
    for move in path:
        if move == '1':
            current_price *= u  # Up move
        else:
            current_price *= d  # Down move
        prices.append(current_price)

    # Compute the arithmetic average of the prices along the path
    avg_price = np.mean(prices)
    average_prices.append(avg_price)


### 3. Payoffs of the Asian option
##### Since we do not have a value for the strike price, we assume that the European Asian call option is at the money, i.e. the strike price is equal the initial stock price.

In [166]:
# Assume strike price is equal to initial price
K = S_0

# Compute payoffs for each terminal node (Asian call option)
payoffs = [max(avg_price - K, 0) for avg_price in average_prices]


In [167]:
# Verifying the number of payoffs at terminal nodes
print(f"Number of terminal payoffs: {len(payoffs)}")
print(f"Expected number of terminal nodes: {2 ** n}")
print("Match:", len(payoffs) == 2 ** n)

Number of terminal payoffs: 33554432
Expected number of terminal nodes: 33554432
Match: True


### 4. Risk-neutral probabilities

In [178]:
p_risk_neutral = (np.exp(r_daily * dt) - d) / (u - d)
q_risk_neutral = 1 - p_risk_neutral

### 5. Backward induction

In [179]:
# Calculate the Asian option price using the risk-neutral probabilities
asian_option_price = 0.0

for j in range(2 ** n):
    path = bin(j)[2:].zfill(n)  # binary path string of length n
    prices = [S_0]
    
    up_moves = 0
    current_price = S_0  # start from S_0

    for move in path:
        if move == '1':
            current_price = current_price * u
            up_moves += 1
        else:
            current_price = current_price * d
        prices.append(current_price)  # record price at each step

    # Compute average of the path (including S_0 and all n steps = n+1 values)
    avg_price = np.mean(prices)

    # Payoff of Asian call option
    payoff = max(avg_price - K, 0)

    # Probability of this path
    down_moves = n - up_moves
    path_prob = (p_risk_neutral ** up_moves) * (q_risk_neutral ** down_moves)

    # Accumulate expected value
    asian_option_price += payoff * path_prob

# Discount back to t=0
asian_option_price *= np.exp(-r * T)
print(f"Asian Call Option Price: {asian_option_price:.2f}")

Asian Call Option Price: 8.65


### 6. Robustness check
#### To analyse the sensitivity of the option, we can change the risk-free rate and the volatility to understand how the price of the option changes

In [ ]:
# === Robustness check grid ===
interest_rates = [0.02, 0.05, 0.10]    # annual r
volatilities   = [0.05, 0.10, 0.20]    # annual σ

results = {}

for r_annual in interest_rates:
    for sigma_annual in volatilities:
        # convert annual σ to daily σ
        sigma_daily = sigma_annual / np.sqrt(250)

        # per-day up/down factors
        u = np.exp(sigma_daily)
        d = np.exp(-sigma_daily)

        # risk-neutral probabilities
        p = (np.exp(r_daily) - d) / (u - d)
        q = 1 - p

        # price under this (r,σ)
        price = 0.0
        for j in range(2 ** n):
            path = bin(j)[2:].zfill(n)
            current = S_0
            ups = 0
            prices = [S_0]
            for bit in path:
                if bit=='1':
                    current *= u
                    ups += 1
                else:
                    current *= d
                prices.append(current)
            avg = np.mean(prices)
            payoff = max(avg - K, 0)
            downs = n - ups
            prob = (p**ups) * (q**downs)
            price += payoff * prob

        # discount 25 days back to today
        price *= np.exp(-r_daily * n)

        results[(r_annual, sigma_annual)] = price
        print(f"r={r_annual:.1%}, sigma={sigma_annual:.1%} → Asian call ≃ {price:.4f}")

# `results` now holds your robustness grid


r=2.0%, sigma=5.0% → Asian call ≃ 1.5177
r=2.0%, sigma=10.0% → Asian call ≃ 2.9349
r=2.0%, sigma=20.0% → Asian call ≃ 5.7705
r=5.0%, sigma=5.0% → Asian call ≃ 1.5177
r=5.0%, sigma=10.0% → Asian call ≃ 2.9349
r=5.0%, sigma=20.0% → Asian call ≃ 5.7705
r=10.0%, sigma=5.0% → Asian call ≃ 1.5177
r=10.0%, sigma=10.0% → Asian call ≃ 2.9349
r=10.0%, sigma=20.0% → Asian call ≃ 5.7705


The effect of the interest rate is practically nonexistent since we are converting the annual interest rate to a daily rate and the period taken into consideration is only 25 days, which is too short to observe any change. 

### 7. Price approximation
#### Here, we use the normal approximation of the binomial distribution to compute the price of the Asian call option

In [204]:
results_normal = {}

for n_approx in [500, 1000]: # For 2 or 4 years of data (250 trading days per year )
    # 1) Mean of the discrete average over n_approx+1 points
    i        = np.arange(n_approx + 1)             
    mean_avg = S_0 * np.exp(r_daily * i).mean()    

    # 2) Variance of that average (n_approx+1 points)
    sigma_avg2 = (sigma_daily**2 / (n_approx + 1)) * \
                 (1 - np.exp(-2 * r_daily * n_approx)) / (2 * r_daily)
    sigma_avg  = np.sqrt(sigma_avg2)

    # 3) Closed‐form normal‐payoff for A ~ N(μ,σ²)
    d      = (mean_avg - K) / sigma_avg
    payoff = (mean_avg - K) * norm.cdf(d) + sigma_avg * norm.pdf(d)

    # 4) Discount back n_approx days at the daily rate
    price_normal_approx = np.exp(-r_daily * n_approx) * payoff

    # Store and print
    results_normal[n_approx] = price_normal_approx
    print(f"n={n_approx:3d} → Normal approx price: {price_normal_approx:.4f}")

# `results_normal` now holds your normal‐approx prices for each n_approx




n=500 → Normal approx price: 3.8410
n=1000 → Normal approx price: 7.5810


Since we only have 25 periods for the binomial model, which are not sufficient for the Central Limit Theorem to "kick-in" when normally approximating the binomial distribution, we can use a new range for n to reflect the 4 years of daily data we are using for this project. 